# Stage C - distillation controls (the scientific heart)

**Question: is it the contrastive objective, or the recipe wrapped around it?**

| Rung | Change | Answers |
|---|---|---|
| C-null | BaCP path, all lambda = 0, one view, no snapshots | **Gate G1.** Does the BaCP code path reproduce plain CE? |
| C0 | CE only, via `pruning_script` | The reference every later rung is measured against |
| C0b | CE + BaCP's two-view augmentation | Is any "BaCP gain" just extra views per step? |
| C1 | + KD (KL on logits) | Does *any* dense-teacher signal help here? |
| C2 | + feature distillation, **same teacher** | Does *feature-level* matching help? |
| C3 | + contrastive form, **same teacher** | Does the *contrastive form* help? |

This is the block the submitted paper had no analogue of at all. It contained no
CE-only arm - "no CE" (lambda_CE = 0) is **not** "CE only" - so nothing separated
the objective from the recipe around it: two views, AutoAugment, the SGD->AdamW
phase swap, the projection head.

**C2 -> C3 is the single most important row in the paper.** Cosine feature matching
is the positive-pair term of InfoNCE with the denominator removed, so with the
teacher held fixed that step isolates one thing: the presence of negatives.

### Gates in this stage
- **G1** (C-null vs C0) - must agree within +/-0.5pp, else the BaCP path is an artefact.
- **G3** (C1, C2 vs C0b) - if neither teacher arm's CI upper bound reaches +0.5pp, STOP.
- **G4** (C3 vs C2) - if the interval contains zero, the contrastive claim has no support.

> **Known defect, not yet fixed:** G1 tests only `abs(C-null - C0) <= 0.5`. If both
> arms collapse to chance (10.0969 on CIFAR-10) it computes `|0.00| <= 0.5` and
> reports **pass** on two dead runs. Check `progress()` for NaN before trusting G1.

> Paired. Every rung shares the same dense teacher, the same optimization steps, and matched forward-pass FLOPs.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))          # so ladder_nb is importable
import ladder_nb as nb
info = nb.setup()


## Configure

`TIER` 1 is the spine (5 seeds). Cells per GPU and dataloader workers are constants at the top of `pool.py`.


In [ ]:
TIER    = 1
GPUS    = info['gpus'] or 1
SEEDS   = None          # None = every seed the tier schedules

RUNGS = ['C-null', 'C0', 'C0b', 'C1', 'C2', 'C3']

import manifest as M
grid = [c for c in M.cells(TIER, rungs=RUNGS)]
print(f'{len(grid)} cell(s) planned over {len(set(c["rung"] for c in grid))} rung(s)')
for r in RUNGS:
    n = sum(1 for c in grid if c['rung'] == r)
    print(f'  {r:10s} {n} seed(s)' if n else f'  {r:10s} -- NOT IN TIER {TIER}')


## Run

Idempotent - a cell is complete iff a record carrying its key exists, so re-running skips what is done. Dense cells run first as a hard barrier. Safe to interrupt; you lose at most the cells in flight.


In [ ]:
summary = nb.run_stage(RUNGS, tier=TIER, gpus=GPUS, seeds=SEEDS)
print(summary['ok'], 'ok,', summary['failed'], 'failed,', summary['skipped'], 'skipped')


## Progress and health

The `nan` column is the one to read first. A diverged run sits at exactly 10.0969% (chance on CIFAR-10) for the rest of training and every delta computed from it is meaningless.


In [ ]:
nb.progress(TIER)


## Watch a single cell

Use this when something looks wrong - it streams one line per epoch so you can see *where* a run breaks rather than only that it did.


In [ ]:
cell = nb.attach(nb.pick('C-null', seed=1, tier=TIER))
nb.show(cell)
# hist, first_nan = nb.watch(cell, gpu=0)
# nb.plot(hist, first_nan, cell['key'])


## Table and gates


In [ ]:
out = nb.report(TIER)
